# ✈️ MSML 610 – Airport Congestion Project  
## Notebook 2: Feature Engineering

This notebook creates the **hourly airport-level congestion dataset** from the raw flight logs.

### Key Outputs:
- Hourly departures  
- Hourly arrivals  
- Total hourly flights  
- Congestion labels (Low / Medium / High)

The output will be saved as:

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parents[0]
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

RAW, PROCESSED

(PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/raw'),
 PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/processed'))

In [11]:
flights = pd.read_csv(RAW / "flights.csv", low_memory=False)
flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
## Convert HHMM times → Hour (0–23)

In [13]:
def hhmm_to_hour(x):
    try:
        x = int(x)
        return x // 100
    except:
        return None

flights["DEP_HOUR"] = flights["SCHEDULED_DEPARTURE"].apply(hhmm_to_hour)
flights["ARR_HOUR"] = flights["SCHEDULED_ARRIVAL"].apply(hhmm_to_hour)

flights[["SCHEDULED_DEPARTURE", "DEP_HOUR", "SCHEDULED_ARRIVAL", "ARR_HOUR"]].head()

,SCHEDULED_DEPARTURE,DEP_HOUR,SCHEDULED_ARRIVAL,ARR_HOUR
0,5,0,430,4
1,10,0,750,7
2,20,0,806,8
3,20,0,805,8
4,25,0,320,3


In [14]:
## Create DATE column for grouping

In [15]:
flights["DATE"] = pd.to_datetime(flights[["YEAR","MONTH","DAY"]])
flights[["YEAR","MONTH","DAY","DATE"]].head()

,YEAR,MONTH,DAY,DATE
0,2015,1,1,2015-01-01
1,2015,1,1,2015-01-01
2,2015,1,1,2015-01-01
3,2015,1,1,2015-01-01
4,2015,1,1,2015-01-01


## Aggregate Hourly Departures & Arrivals
We compute:
- number of departures per airport-hour
- number of arrivals per airport-hour

In [19]:
# Departures
dep = (flights.groupby(["ORIGIN_AIRPORT","DATE","DEP_HOUR"])
       .size().reset_index(name="departures"))
dep.rename(columns={"ORIGIN_AIRPORT":"AIRPORT","DEP_HOUR":"HOUR"}, inplace=True)

# Arrivals
arr = (flights.groupby(["DESTINATION_AIRPORT","DATE","ARR_HOUR"])
       .size().reset_index(name="arrivals"))
arr.rename(columns={"DESTINATION_AIRPORT":"AIRPORT","ARR_HOUR":"HOUR"}, inplace=True)

dep.head(), arr.head()

(  AIRPORT       DATE  HOUR  departures
 0   10135 2015-10-01     6           3
 1   10135 2015-10-01    12           3
 2   10135 2015-10-01    16           1
 3   10135 2015-10-01    17           2
 4   10135 2015-10-02     6           3,
   AIRPORT       DATE  HOUR  arrivals
 0   10135 2015-10-01    11         2
 1   10135 2015-10-01    12         1
 2   10135 2015-10-01    15         1
 3   10135 2015-10-01    16         2
 4   10135 2015-10-01    21         1)

In [22]:
## Merge Departures + Arrivals

In [25]:
hourly = pd.merge(dep, arr, on=["AIRPORT","DATE","HOUR"], how="outer")
hourly.head()

,AIRPORT,DATE,HOUR,departures,arrivals
0,10135,2015-10-01,6,3.0,NaN
1,10135,2015-10-01,11,NaN,2.0
2,10135,2015-10-01,12,3.0,1.0
3,10135,2015-10-01,15,NaN,1.0
4,10135,2015-10-01,16,1.0,2.0


In [28]:
hourly["departures"] = hourly["departures"].fillna(0).astype(int)
hourly["arrivals"] = hourly["arrivals"].fillna(0).astype(int)
hourly["total_flights"] = hourly["departures"] + hourly["arrivals"]

hourly.head()

,AIRPORT,DATE,HOUR,departures,arrivals,total_flights
0,10135,2015-10-01,6,3,0,3
1,10135,2015-10-01,11,0,2,2
2,10135,2015-10-01,12,3,1,4
3,10135,2015-10-01,15,0,1,1
4,10135,2015-10-01,16,1,2,3


## Label Congestion Levels

We create 3 bins:

- **Low**: 0–20 flights  
- **Medium**: 21–50 flights  
- **High**: 51+ flights  

In [33]:
hourly["congestion_level"] = pd.cut(
    hourly["total_flights"],
    bins=[-1, 20, 50, hourly["total_flights"].max()+1],
    labels=["Low", "Medium", "High"]
)

hourly.head()

,AIRPORT,DATE,HOUR,departures,arrivals,total_flights,congestion_level
0,10135,2015-10-01,6,3,0,3,Low
1,10135,2015-10-01,11,0,2,2,Low
2,10135,2015-10-01,12,3,1,4,Low
3,10135,2015-10-01,15,0,1,1,Low
4,10135,2015-10-01,16,1,2,3,Low


In [36]:
## Save the processed dataset

In [39]:
output_path = PROCESSED / "hourly_congestion.csv"
hourly.to_csv(output_path, index=False)

output_path

PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/processed/hourly_congestion.csv')

# 🎉 Feature Engineering Complete

The dataset is ready for modeling in `03_training_xgboost.ipynb`.